# PostgreSQL: joins and aggregations

**ZHAW Data Analytics HS2026 · Group project “Electricity prices of the municipalities” · Part Metehan (Bonus 2)**

This notebook transfers the SQLite database from `scripts/02_sqlite_metehan.ipynb` to PostgreSQL and analyses it there with joins and aggregations.

**Why PostgreSQL in addition?** SQLite is a single file without a server. PostgreSQL is a real database server, as used in companies, and can do more:

| | SQLite | PostgreSQL |
| --- | --- | --- |
| Operation | file, no server | server, several users at the same time |
| Storing prices | `REAL` (floating point, small rounding errors) | `NUMERIC(7,3)` (exact to 3 decimal places) |
| Median | missing, had to be calculated in pandas | `percentile_cont(0.5)` |
| Statistics in SQL | only `AVG`, `MIN`, `MAX` | also `STDDEV`, `CORR` and others |
| Wide format | `MAX(CASE WHEN ...)` | `MAX(...) FILTER (WHERE ...)` |

**Requirements**

1. PostgreSQL is installed and running (installer from postgresql.org or Postgres.app on the Mac).
2. An empty database exists: in the terminal `createdb -U postgres strompreise`, or in pgAdmin via “Create Database”.
3. The Python driver is installed: `pip install psycopg2-binary`.
4. `sql/strompreise.db` from notebook 02 is in the repository root.

The password is not in the code and there is no interactive prompt. Connections are configured via the environment variables `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER` and `PGPASSWORD`.

*Identifiers stay German: table, column, view and variable names are shared with the CSV files, the SQLite and PostgreSQL databases and the other notebooks, so only text and comments are translated.*

In [1]:
import os
import sqlite3
from decimal import Decimal
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from pathlib import Path

def finde_wurzel(start):
    pfad = Path(start).resolve()
    # Same criterion as in the notebooks in scripts/: daten/ and scripts/ side by side
    for kandidat in (pfad, *pfad.parents):
        if (kandidat / "daten").is_dir() and (kandidat / "scripts").is_dir():
            return kandidat
    raise FileNotFoundError("Repository root not found")

WURZEL = finde_wurzel(Path.cwd())
SQLITE_DB = WURZEL / "sql" / "strompreise.db"
if not SQLITE_DB.exists():
    raise FileNotFoundError(f"SQLite database missing: {SQLITE_DB}")

if not os.environ.get("PGPASSWORD"):
    raise RuntimeError(
        "PGPASSWORD is missing. Set PGPASSWORD before the automated notebook run; "
        "there is no interactive password prompt."
    )

PG = {
    "host": os.environ.get("PGHOST", "localhost"),
    "port": int(os.environ.get("PGPORT", "5432")),
    "dbname": os.environ.get("PGDATABASE", "strompreise"),
    "user": os.environ.get("PGUSER", "postgres"),
    "password": os.environ["PGPASSWORD"],
}

con = psycopg2.connect(**PG)
lite = sqlite3.connect(SQLITE_DB)   # Source: database from notebook 02


def sql(befehl):
    # Run an SQL command without a result (CREATE, DROP, INSERT ...)
    with con.cursor() as cur:
        cur.execute(befehl)
    con.commit()


def abfrage(befehl):
    # Run a SELECT and return it as a DataFrame.
    # NUMERIC comes from PostgreSQL as Decimal and is converted to float for pandas.
    with con.cursor() as cur:
        cur.execute(befehl)
        spalten = [d[0] for d in cur.description]
        zeilen = cur.fetchall()
    df = pd.DataFrame(zeilen, columns=spalten)
    for c in df.columns:
        if df[c].map(lambda v: isinstance(v, Decimal)).any():
            df[c] = df[c].astype(float)
    return df


abfrage("SELECT version()")

,version
0,"PostgreSQL 17.12 on x86_64-windows, compiled b..."


## 1 · Create the tables

The schema is the same as in SQLite. Only the data types have changed: prices are `NUMERIC(7,3)`, i.e. exact decimal numbers with three decimal places, as supplied by ElCom.

`DROP ... CASCADE` first deletes old tables and views. This way the notebook can be rerun as often as needed.

In [2]:
sql('''
DROP VIEW  IF EXISTS v_analyse, v_preis_h4 CASCADE;
DROP TABLE IF EXISTS kennzahl, indikator, tarif, netzbetreiber, gemeinde CASCADE;

CREATE TABLE gemeinde (
    gemeinde_nr      INTEGER PRIMARY KEY,
    name             TEXT
);

CREATE TABLE netzbetreiber (
    netzbetreiber_id INTEGER PRIMARY KEY,
    name             TEXT NOT NULL
);

CREATE TABLE tarif (
    jahr               SMALLINT     NOT NULL,
    gemeinde_nr        INTEGER      NOT NULL REFERENCES gemeinde(gemeinde_nr),
    netzbetreiber_id   INTEGER      NOT NULL REFERENCES netzbetreiber(netzbetreiber_id),
    produkt            TEXT         NOT NULL CHECK (produkt IN ('standard', 'cheapest')),
    kategorie          TEXT         NOT NULL,
    total              NUMERIC(7,3) NOT NULL,      -- Rp./kWh excl. VAT
    energie            NUMERIC(7,3),
    netznutzung        NUMERIC(7,3),
    abgaben            NUMERIC(7,3),
    netzzuschlag       NUMERIC(7,3),
    messtarif          NUMERIC(7,3) NOT NULL DEFAULT 0,
    messtarif_chf_jahr NUMERIC(9,3),
    PRIMARY KEY (jahr, gemeinde_nr, netzbetreiber_id, produkt, kategorie)
);

CREATE TABLE indikator (
    code         TEXT PRIMARY KEY,
    bezeichnung  TEXT NOT NULL,
    einheit      TEXT,
    periode      TEXT,
    label_quelle TEXT CHECK (label_quelle IN ('geprueft', 'abgeleitet'))
);

CREATE TABLE kennzahl (
    gemeinde_nr INTEGER NOT NULL REFERENCES gemeinde(gemeinde_nr),
    indikator   TEXT    NOT NULL REFERENCES indikator(code),
    wert        DOUBLE PRECISION,
    status      TEXT,
    PRIMARY KEY (gemeinde_nr, indikator)
);

CREATE INDEX idx_tarif_produkt_kat ON tarif (produkt, kategorie, jahr);
''')
abfrage('''
    SELECT table_name AS tabelle, COUNT(*) AS spalten
    FROM information_schema.columns
    WHERE table_schema = 'public'
    GROUP BY table_name
    ORDER BY table_name
''')

,tabelle,spalten
0,gemeinde,2
1,indikator,5
2,kennzahl,4
3,netzbetreiber,2
4,tarif,12


## 2 · Take over the data from SQLite

The data comes directly from `strompreise.db`. This guarantees that SQLite and PostgreSQL work with the same rows. The order matters: `gemeinde`, `netzbetreiber` and `indikator` come first, because `tarif` and `kennzahl` refer to them.

`execute_values` sends many rows in one command. This is much faster than one `INSERT` per row.

In [3]:
TABELLEN = ["gemeinde", "netzbetreiber", "indikator", "tarif", "kennzahl"]

for tabelle in TABELLEN:
    spalten = [r[1] for r in lite.execute(f"PRAGMA table_info({tabelle})")]
    zeilen = lite.execute(f"SELECT {', '.join(spalten)} FROM {tabelle}").fetchall()
    with con.cursor() as cur:
        execute_values(cur, f"INSERT INTO {tabelle} ({', '.join(spalten)}) VALUES %s", zeilen, page_size=5000)
    con.commit()
    print(f"{tabelle:<14} {len(zeilen):>8,} rows transferred")

# Update the statistics for the query planner right away. Without this step,
# PostgreSQL plans the queries below with outdated information about the freshly loaded tables;
# query 4.6 then ran for over ten minutes instead of under a second.
sql("ANALYZE " + ", ".join(TABELLEN))

gemeinde          2,225 rows transferred
netzbetreiber       609 rows transferred
indikator            41 rows transferred


tarif           143,837 rows transferred


kennzahl         89,052 rows transferred


**Check:** every table must have as many rows in PostgreSQL as in SQLite, and the sum of the price components must equal the total.

In [4]:
vergleich = pd.DataFrame({
    "tabelle": TABELLEN,
    "sqlite": [lite.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in TABELLEN],
    "postgresql": [abfrage(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in TABELLEN],
})
vergleich["gleich"] = vergleich["sqlite"] == vergleich["postgresql"]
vergleich

,tabelle,sqlite,postgresql,gleich
0,gemeinde,2225,2225,True
1,netzbetreiber,609,609,True
2,indikator,41,41,True
3,tarif,143837,143837,True
4,kennzahl,89052,89052,True


In [5]:
abfrage('''
    SELECT jahr,
           COUNT(*) AS zeilen,
           COUNT(*) FILTER (
               WHERE ABS(total - (energie + netznutzung + abgaben + netzzuschlag + messtarif)) > 0.01
           ) AS summe_falsch
    FROM tarif
    GROUP BY jahr
    ORDER BY jahr
''')

,jahr,zeilen,summe_falsch
0,2021,54210,0
1,2024,48187,0
2,2027,41440,0


## 3 · Views

The two views are the same as in SQLite. PostgreSQL writes the pivot from long to wide format more briefly with `FILTER (WHERE ...)` instead of `CASE WHEN`.

**Placeholder rows.** In 2021, four grid operators reported H4 rows with a grid usage charge of 0. An H4 household tariff without grid usage cannot exist, and the same operators report real grid usage in 2024 and 2027. Where such a row sits next to a real tariff in the same municipality and year, `v_preis_h4` leaves it out (10 rows), otherwise the mean would halve the price. Sufers, where the zero row is the only one, is kept so that the analysis set stays at 2,087 municipalities. The table `tarif` is not changed; see `dokumentation_integration_lenny.md`, section 6b.

In [6]:
sql('''
CREATE VIEW v_preis_h4 AS
SELECT jahr,
       gemeinde_nr,
       COUNT(*)                     AS anzahl_netzbetreiber,
       AVG(total)                   AS total,
       AVG(energie)                 AS energie,
       AVG(netznutzung + messtarif) AS netznutzung_inkl_mess,
       AVG(abgaben)                 AS abgaben,
       AVG(netzzuschlag)            AS netzzuschlag
FROM tarif t
WHERE produkt = 'standard' AND kategorie = 'H4'
  -- Placeholder rows (2021): grid usage 0 next to a real tariff of another operator
  -- in the same municipality and year. They would halve the mean, so they are left
  -- out here. A zero row that is the municipality's only row (Sufers) stays, so that
  -- no municipality drops out. The table tarif itself is unchanged.
  AND NOT (t.netznutzung = 0 AND EXISTS (
        SELECT 1 FROM tarif t2
        WHERE t2.jahr = t.jahr AND t2.gemeinde_nr = t.gemeinde_nr
          AND t2.produkt = 'standard' AND t2.kategorie = 'H4'
          AND t2.netzbetreiber_id <> t.netzbetreiber_id
          AND t2.netznutzung <> 0))
GROUP BY jahr, gemeinde_nr;

CREATE VIEW v_analyse AS
WITH merkmale AS (
    SELECT gemeinde_nr,
           MAX(wert) FILTER (WHERE indikator = 'Ind_01_01') AS einwohner,
           MAX(wert) FILTER (WHERE indikator = 'Ind_01_03') AS dichte,
           MAX(wert) FILTER (WHERE indikator = 'Ind_04_01') AS flaeche_total,
           MAX(wert) FILTER (WHERE indikator = 'Ind_06_03') AS beschaeftigte,
           MAX(wert) FILTER (WHERE indikator = 'Ind_01_06') AS anteil_65plus
    FROM kennzahl
    GROUP BY gemeinde_nr
)
SELECT g.gemeinde_nr,
       g.name     AS gemeinde,
       p21.anzahl_netzbetreiber,
       p21.total  AS total_2021,
       p24.total  AS total_2024,
       p27.total  AS total_2027,
       m.einwohner, m.dichte, m.flaeche_total, m.beschaeftigte, m.anteil_65plus
FROM gemeinde g
JOIN v_preis_h4 p21 ON p21.gemeinde_nr = g.gemeinde_nr AND p21.jahr = 2021
JOIN v_preis_h4 p24 ON p24.gemeinde_nr = g.gemeinde_nr AND p24.jahr = 2024
JOIN v_preis_h4 p27 ON p27.gemeinde_nr = g.gemeinde_nr AND p27.jahr = 2027
JOIN merkmale  m   ON m.gemeinde_nr   = g.gemeinde_nr;
''')
abfrage("SELECT COUNT(*) AS gemeinden FROM v_analyse")

,gemeinden
0,2087


## 4 · Analyses with joins and aggregations

**4.1 · Distribution of the H4 prices per year.** PostgreSQL calculates the median and standard deviation directly. In SQLite the median still needed pandas.

In [7]:
abfrage('''
    SELECT jahr,
           COUNT(*)                                                   AS gemeinden,
           ROUND(AVG(total), 2)                                       AS mittelwert,
           ROUND(percentile_cont(0.5) WITHIN GROUP (ORDER BY total)::numeric, 2) AS median,
           ROUND(STDDEV(total), 2)                                    AS std_abw,
           ROUND(MIN(total), 2)                                       AS minimum,
           ROUND(MAX(total), 2)                                       AS maximum
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''')

,jahr,gemeinden,mittelwert,median,std_abw,minimum,maximum
0,2021,2201,20.90,20.63,2.93,8.59,27.65
1,2024,2140,32.00,32.15,3.73,10.22,50.62
2,2027,2109,27.24,26.91,3.06,10.56,45.88


**4.2 · Share of the components in the total.** The share shows which part of the price carried the increase. The column `netznutzung_inkl_mess` makes the years comparable, because the metering tariff is reported separately from 2026.

In [8]:
abfrage('''
    SELECT jahr,
           ROUND(AVG(total), 2)                                      AS total,
           ROUND(AVG(energie), 2)                                    AS energie,
           ROUND(AVG(netznutzung_inkl_mess), 2)                      AS netznutzung,
           ROUND(AVG(abgaben + netzzuschlag), 2)                     AS abgaben_zuschlag,
           ROUND(100 * SUM(energie) / SUM(total), 1)                 AS anteil_energie_pct,
           ROUND(100 * SUM(netznutzung_inkl_mess) / SUM(total), 1)   AS anteil_netz_pct
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''')

,jahr,total,energie,netznutzung,abgaben_zuschlag,anteil_energie_pct,anteil_netz_pct
0,2021,20.90,7.96,9.79,3.15,38.1,46.8
1,2024,32.00,15.59,13.22,3.18,48.7,41.3
2,2027,27.24,11.19,12.86,3.20,41.1,47.2


**4.3 · The most expensive larger grid operators in 2027.** Here `tarif` and `netzbetreiber` are joined, aggregated per operator and restricted with `HAVING` to operators with at least 10 municipalities. `RANK()` is a window function; it numbers the rows without combining them.

In [9]:
abfrage('''
    SELECT RANK() OVER (ORDER BY AVG(t.total) DESC) AS rang,
           n.name                                   AS netzbetreiber,
           COUNT(DISTINCT t.gemeinde_nr)            AS gemeinden,
           ROUND(AVG(t.total), 2)                   AS preis_2027
    FROM tarif t
    JOIN netzbetreiber n ON n.netzbetreiber_id = t.netzbetreiber_id
    WHERE t.jahr = 2027 AND t.produkt = 'standard' AND t.kategorie = 'H4'
    GROUP BY n.netzbetreiber_id, n.name
    HAVING COUNT(DISTINCT t.gemeinde_nr) >= 10
    ORDER BY rang
    LIMIT 10
''')

,rang,netzbetreiber,gemeinden,preis_2027
0,1,Primeo Netz AG,64,31.33
1,2,Elektrizitätswerk des Kantons Schaffhausen AG,26,30.82
2,3,Repower AG,33,29.88
3,4,AEW Energie AG,75,29.75
4,5,St. Gallisch-Appenzellische Kraftwerke AG,40,29.45
5,6,Società Elettrica Sopracenerina SA (Ticino),36,28.80
6,7,"Genossenschaft Elektra, Jegenstorf",22,28.78
7,8,Romande Energie SA,262,28.36
8,9,Société des Forces Électriques de La Goule SA,16,28.04
9,10,EBL (Genossenschaft Elektra Baselland),50,27.94


**4.4 · Does the price fall back everywhere?** The window function `LAG` fetches the price of the previous period for each municipality. Then we count where the price fell in 2027 and where it is still above the 2021 level. This is a first answer to the question from the project plan “whether it does not fall back everywhere”.

In [10]:
abfrage('''
    WITH verlauf AS (
        SELECT gemeinde_nr, jahr, total,
               LAG(total) OVER (PARTITION BY gemeinde_nr ORDER BY jahr) AS vorher,
               FIRST_VALUE(total) OVER (PARTITION BY gemeinde_nr ORDER BY jahr) AS stand_2021
        FROM v_preis_h4
        WHERE gemeinde_nr IN (SELECT gemeinde_nr FROM v_analyse)   -- only municipalities with all 3 years
    )
    SELECT COUNT(*)                                              AS gemeinden,
           COUNT(*) FILTER (WHERE total < vorher)                AS guenstiger_als_2024,
           COUNT(*) FILTER (WHERE total >= vorher)               AS gleich_oder_teurer_als_2024,
           COUNT(*) FILTER (WHERE total <= stand_2021)           AS zurueck_auf_2021,
           ROUND(AVG(total - vorher), 2)                         AS mittl_aenderung_24_27,
           ROUND(AVG(total - stand_2021), 2)                     AS mittl_aenderung_21_27
    FROM verlauf
    WHERE jahr = 2027
''')

,gemeinden,guenstiger_als_2024,gleich_oder_teurer_als_2024,zurueck_auf_2021,mittl_aenderung_24_27,mittl_aenderung_21_27
0,2087,1922,165,227,-4.74,6.35


**4.5 · Relationship with municipal characteristics.** `CORR` calculates Pearson's correlation coefficient directly in SQL. For this, prices and indicators are joined. Population and density are log-transformed (`LN`), because otherwise a few large cities dominate the values.

This is only a first overview without p-values. Subraj does the tests with p-values in the analysis (Bonus 4).

In [11]:
abfrage('''
    SELECT p.jahr,
           COUNT(*)                                      AS gemeinden,
           ROUND(CORR(p.total, LN(m.einwohner))::numeric, 3)  AS r_ln_einwohner,
           ROUND(CORR(p.total, LN(m.dichte))::numeric, 3)     AS r_ln_dichte,
           ROUND(CORR(p.total, m.anteil_65plus)::numeric, 3)  AS r_anteil_65plus,
           ROUND(CORR(p.total, p.anzahl_netzbetreiber)::numeric, 3) AS r_anzahl_betreiber
    FROM v_preis_h4 p
    JOIN v_analyse m ON m.gemeinde_nr = p.gemeinde_nr
    GROUP BY p.jahr
    ORDER BY p.jahr
''')

,jahr,gemeinden,r_ln_einwohner,r_ln_dichte,r_anteil_65plus,r_anzahl_betreiber
0,2021,2087,-0.159,-0.163,0.238,-0.004
1,2024,2087,0.115,0.205,-0.106,0.078
2,2027,2087,0.031,0.070,-0.064,0.093


**4.6 · Prices by population quartile.** `NTILE(4)` divides the municipalities into four equally sized groups by population. With `FILTER`, each year is then in its own column.

In [12]:
abfrage('''
    WITH quartile AS (
        SELECT gemeinde_nr, NTILE(4) OVER (ORDER BY einwohner) AS quartil
        FROM v_analyse
    )
    SELECT q.quartil,
           COUNT(DISTINCT q.gemeinde_nr)                         AS gemeinden,
           MIN(a.einwohner)::int                                 AS einwohner_von,
           MAX(a.einwohner)::int                                 AS einwohner_bis,
           ROUND(AVG(p.total) FILTER (WHERE p.jahr = 2021), 2)   AS preis_2021,
           ROUND(AVG(p.total) FILTER (WHERE p.jahr = 2024), 2)   AS preis_2024,
           ROUND(AVG(p.total) FILTER (WHERE p.jahr = 2027), 2)   AS preis_2027
    FROM quartile q
    JOIN v_analyse  a ON a.gemeinde_nr = q.gemeinde_nr
    JOIN v_preis_h4 p ON p.gemeinde_nr = q.gemeinde_nr
    GROUP BY q.quartil
    ORDER BY q.quartil
''')

,quartil,gemeinden,einwohner_von,einwohner_bis,preis_2021,preis_2024,preis_2027
0,1,522,32,764,21.69,31.36,26.99
1,2,522,769,1627,20.94,32.04,27.41
2,3,522,1629,3924,20.70,31.96,27.45
3,4,521,3932,420217,20.28,32.59,27.16


## 5 · Check against SQLite

The analysis table from PostgreSQL must return the same municipalities and values as the one from SQLite. Small differences in the last digit are possible, because SQLite calculates with floating-point numbers and PostgreSQL exactly.

In [13]:
aus_pg = abfrage("SELECT * FROM v_analyse").set_index("gemeinde_nr").sort_index()
aus_lite = pd.read_sql("SELECT * FROM v_analyse", lite).set_index("gemeinde_nr").sort_index()

print("Municipalities PostgreSQL / SQLite:", len(aus_pg), "/", len(aus_lite))
print("Same municipality numbers:   ", aus_pg.index.equals(aus_lite.index))
for spalte in ["total_2021", "total_2024", "total_2027", "einwohner", "dichte", "beschaeftigte"]:
    diff = (aus_pg[spalte].astype(float) - aus_lite[spalte]).abs().max()
    print(f"{spalte:<15} largest deviation: {diff:.2e}")

Municipalities PostgreSQL / SQLite: 2087 / 2087
Same municipality numbers:    True
total_2021      largest deviation: 3.55e-15
total_2024      largest deviation: 7.11e-15
total_2027      largest deviation: 7.11e-15
einwohner       largest deviation: 0.00e+00
dichte          largest deviation: 0.00e+00
beschaeftigte   largest deviation: 0.00e+00


## 6 · Conclusion

- Both databases contain the same data and return the same analysis table. The checks above show this.
- PostgreSQL is worthwhile above all for analyses: median, standard deviation, correlation and window functions such as `LAG`, `RANK` and `NTILE` run directly in SQL.
- Nothing changes for the group. Subraj can keep working with `v_analyse` from SQLite; the numbers are identical.

In [14]:
con.close()
lite.close()